# Triton Refill: UCSD Water Station Finder

This project helps students find nearby water stations at UC San Diego.

The notebook prepares a saved station list, tries a nearby search, and saves the results for use in ArcGIS. Read the explanation above each code block, then run that block before moving to the next one.

| Step | What happens |
| --- | --- |
| Open | Read the saved list of stations. |
| Clean | Tidy names, label missing information, and check for repeated entries. |
| Find | Measure distances from an example starting point and show nearby stations. |
| Check | Try searches with predictable answers to catch mistakes. |
| Save | Make files that can be opened in other programs. |

A **table** is like a spreadsheet. Each **row** is one saved entry. Each **column** holds one kind of information, such as a name or location.

Names in the code, such as `stations`, are labels for information the computer keeps while the notebook runs. An equals sign usually means “save this under that name.”

All station data is already included here. The separate ArcGIS app handles the interactive map and the user's current location.


## 1. Get the tools ready

This block gets the tools Python needs to read the station list and measure distances. Think of it as getting your supplies ready before starting a task.

- The first line installs **GeoPy**, a tool for working with locations. `2.4.1` chooses a specific version; the other options reduce installation messages and avoid using a saved installer.
- `import pandas as pd` loads **pandas**, which works with tables. `pd` is its short name in this notebook.
- `StringIO` lets the computer read the station text stored below as though it came from a file.
- `geodesic` is the GeoPy tool that will measure distance between two map points.

`import` means “make this tool available.” At this stage, the notebook has only prepared its tools. The station list is loaded in the following blocks.


In [ ]:
%pip -q install --no-cache-dir geopy==2.4.1

import pandas as pd
from io import StringIO
from geopy.distance import geodesic

## 2. Keep the station list here

This block contains the saved station information. The long text lists places, descriptions, and map locations. You can understand the block without memorizing every entry.

The name `csv_data` holds this text. The three quotation marks at each end let the text continue across many lines.

The first line names the columns:

| Column | Meaning |
| --- | --- |
| `station_name` | The place or station name. |
| `description` | Extra details, such as the floor or nearby landmark. |
| `latitude` | The location's north–south position. |
| `longitude` | The location's east–west position. |
| `source_group` | The group the entry came from in the original map. |

Latitude and longitude work together to identify a point on Earth. Each following line describes one entry. A blank description means the source did not supply those details.

This is a saved copy of the source map's information. Running the notebook reads this copy; it does not fetch the latest station information. [1]


In [ ]:
# @title Saved water-station list
# This text keeps the data inside the notebook.
csv_data = """station_name,description,latitude,longitude,source_group
The Zone,,32.880002,-117.236862,Hydration Locations
The Sustainability Resource Center,,32.879943,-117.237124,Hydration Locations
Price Theater,,32.879993,-117.237055,Hydration Locations
Price Center - 2nd Floor,Near PC West hallway entrance,32.879695,-117.236588,Hydration Locations
Price Center - 4th Floor,Next to the elevator (PC East),32.879677,-117.236406,Hydration Locations
Porters Pub,Foyer (Currently closed),32.8770015,-117.239479,Hydration Locations
Geisel 1st Floor,There are 4 hydration stations in Geisel,32.8811281,-117.2377258,Hydration Locations
Hi Thai,,32.87719,-117.240214,Hydration Locations
Peterson Hall,,32.880004,-117.240043,Hydration Locations
Galbraith Hall,There are three hydration stations in Galbraith Hall,32.874028,-117.240976,Hydration Locations
CS&E Building,Student Lounge,32.881862,-117.23345,Hydration Locations
Pepper Canyon ,between the bathrooms,32.878209,-117.233954,Hydration Locations
Campus Services Complex Building C,,32.882948,-117.229625,Hydration Locations
Biology Field Station,,32.885561,-117.230161,Hydration Locations
EBU-II,There are hydration stations on every floor,32.881155,-117.233284,Hydration Locations
Recreation Gym,,32.876758,-117.241185,Hydration Locations
Main Gym,,32.877001,-117.241169,Hydration Locations
Dance Studio Facility,,32.872811,-117.238991,Hydration Locations
Medical Teaching Facility,There are two hydration stations here,32.875668,-117.235386,Hydration Locations
HSS,,32.878371,-117.241679,Hydration Locations
Price Center - 3rd Floor,By the restrooms,32.8797696,-117.2357136,Hydration Locations
AP&M 1st Floor Lobby,,32.8792673,-117.2409801,Hydration Locations
AP&M 2nd Floor Lobby,,32.8792673,-117.2409801,Hydration Locations
RIMAC 1,There are 3 hydration stations in RIMAC,32.8852206,-117.2403967,Hydration Locations
RIMAC 2,There are 3 hydration stations in RIMAC,32.8852713,-117.2396484,Hydration Locations
RIMAC 3,There are 3 hydration stations in RIMAC,32.885381,-117.239324,Hydration Locations
Biomed Library,1st Floor Lobby,32.8755347,-117.2366631,Hydration Locations
Canyonview Pool,Hydration station located by locker rooms,32.8807225,-117.2316957,Hydration Locations
Geisel 6th Floor,There are 4 hydration stations in Geisel,32.8811088,-117.2375804,Hydration Locations
Geisel 2nd Floor,There are 4 hydration stations in Geisel,32.8811392,-117.2374047,Hydration Locations
Geisel 8th Floor,There are 4 hydration stations in Geisel,32.881111,-117.2375549,Hydration Locations
Telemed 1st Floor,There are 2 hydration stations in Telemed,32.8752712,-117.2346246,Hydration Locations
Telemed 3rd Floor,There are 2 hydration stations in Telemed,32.8753252,-117.2347319,Hydration Locations
Price Center-1st Floor,Next to Burger King,32.879988,-117.235832,Hydration Locations
Warren Commuter Lounge,,32.8814365,-117.2338817,Hydration Locations
Center Hall,1st Floor - Near the Restrooms,32.87815,-117.236824,Hydration Locations
Student Services Center,1st Floor - Near the stair case and restroom,32.8787087,-117.2356385,Hydration Locations
SIO Administration Building,,32.8649974,-117.254221,Hydration Locations
Deep Sea Drilling West ,,32.86754,-117.251647,Hydration Locations
Eckart Building ,Second Floor ,32.867141,-117.252537,Hydration Locations
Student Activity Center,,32.882403,-117.233182,HDH Hydration Locations
Dogg House,,32.8778572,-117.2332594,HDH Hydration Locations
Warren Residential Life Office,,32.8840216,-117.2333439,HDH Hydration Locations
The Village Residential Life Office,Outside the Lodge,32.8781,-117.230301,HDH Hydration Locations
ERC Great Hall,downstairs between the bathrooms,32.883966,-117.2419363,HDH Hydration Locations
Marshall's Residential Life Office,,32.8831773,-117.2428181,HDH Hydration Locations
Fireside Lounge,,32.8821776,-117.2387713,HDH Hydration Locations
ERC Laundry South,,32.884637,-117.243122,HDH Hydration Locations
Goldberg Hall Laundry Room,Laundry Room ,32.8830966,-117.2338361,HDH Hydration Locations
Argo Hall,,32.8744512,-117.2415447,HDH Hydration Locations
Muir Across from Roots,between the bathrooms,32.8790153,-117.2425509,HDH Hydration Locations
Muir - Roots,,32.8789866,-117.242615,HDH Hydration Locations
Pines,,32.8790009,-117.2424476,HDH Hydration Locations
Goody's Market,,32.882921,-117.240461,HDH Hydration Locations
Blake Hall,,32.875037,-117.241523,HDH Hydration Locations
OceanView Terrace Restaurant,,32.883182,-117.242714,HDH Hydration Locations
Cafe Ventanas,,32.88611,-117.242661,HDH Hydration Locations
Canyon Vista,Inside next to the television and outside next to the main entrance.,32.8839864,-117.2330922,HDH Hydration Locations
Foodworx,,32.878812,-117.23043,HDH Hydration Locations
Seventh College East,The hydration center is at the outside laundry room aka The Soap Bar,32.8882166,-117.2418723,HDH Hydration Locations
Tamarack Apartments,Between bathrooms behind John's Market,32.8782233,-117.2423279,HDH Hydration Locations
Black Hall Laundry Room,Laundry Room ,32.8821235,-117.232672,HDH Hydration Locations
The Village Apartments Laundry Room,Building 300,32.8788213,-117.229405,HDH Hydration Locations
Seventh College Tower West ,Hydration station located in the 15th floor conference room,32.8886127,-117.2422528,HDH Hydration Locations
Matthews Building B,Laundry Room,32.8786974,-117.2308002,HDH Hydration Locations
Seventh College West Building 3,Laundry Room,32.8874753,-117.2427759,HDH Hydration Locations
ERC East Upper Laundry Room,by Kathmandu House,32.8852815,-117.2418841,HDH Hydration Locations
ERC POP Offices,Between bathrooms upstairs of Cafe V,32.8859831,-117.2425184,HDH Hydration Locations
Marshall Uppers Laundry Room,Building J Near hammock lounge,32.8830696,-117.2415823,HDH Hydration Locations
P Building Res Halls,Laundry Room,32.8835268,-117.2429314,HDH Hydration Locations
V Building Res Hall,Laundry room,32.8830223,-117.2431353,HDH Hydration Locations
Keeling Apartment Building 2,Ground floor facing south basketball courts,32.8737258,-117.2432493,HDH Hydration Locations
Revelle - Rogers Market,Hallway by Amazon lockers,32.8745143,-117.2423279,HDH Hydration Locations
Rita Atkinson Residences,4th floor gym,32.8727549,-117.2350645,HDH Hydration Locations
Q Building Res Hall,Laundry room,32.8835516,-117.243138,HDH Hydration Locations
Mesa Nueva,2 by the leasing office,32.8756316,-117.2228229,HDH Hydration Locations
Mesa Nueva - Fitness Center,,32.8750003,-117.2235953,HDH Hydration Locations
Mesa Apartments - Bldg 9156,Laundry Building. In the nook across from the restrooms.,32.8721506,-117.2221383,HDH Hydration Locations
Seventh College West Building 2,Downstairs by the bathroom,32.8880586,-117.2424969,HDH Hydration Locations
OMS Leasing Office/Lounge,"Outside, on the first floor",32.873865,-117.2257905,HDH Hydration Locations
Nuevo West,1 near Fitness Center. 1 near Game Room restrooms.,32.8745228,-117.2248646,HDH Hydration Locations
Nuevo East,2 at Exchange Building. ,32.8746414,-117.2186881,HDH Hydration Locations
Coast,1 in Laundry Room,32.8701292,-117.2469013,HDH Hydration Locations
Nuevo East,1 at the Fitness Center,32.8750448,-117.219354,HDH Hydration Locations
"""

## 3. Turn the list into a table

This block organizes the saved text into rows and columns, so the computer can work with station names, descriptions, and locations separately.

1. `pd.read_csv(...)` reads the text as a table. CSV is a simple file format that stores table values separated by commas. `StringIO(csv_data)` supplies the text from the previous block.
2. `raw` keeps the original table.
3. `stations = raw.copy()` makes a separate working copy. Later cleanup changes happen to `stations`, so the original remains available for comparison.
4. `stations.head()` shows the first five rows as a quick preview.

Showing five rows does not remove the others. The full table is still there. If the preview shows `NaN`, that means information is missing.


In [ ]:
raw = pd.read_csv(StringIO(csv_data))
stations = raw.copy()

stations.head()

## 4. Count entries and missing information

Before cleaning anything, this block checks how much information the original list contains and where it has gaps.

`len(raw)` counts the rows. The answer is saved as `original_count` so it can be used again later.

`raw["description"]` selects the description column. The next two parts work together:

- `.isna()` asks “Is this value missing?” for each row.
- `.sum()` counts the yes answers. The computer counts each yes as 1 and each no as 0.

For example, if two of five descriptions are missing, this produces a count of 2.

`print(...)` displays the counts. The last line checks every column for missing values.

The saved results show **84 entries and 31 missing descriptions**. These are counts from the source list; some entries describe places with several stations.


In [ ]:
original_count = len(raw)
missing_descriptions = raw["description"].isna().sum()

print("Mapped entries:", original_count)
print("Missing descriptions:", missing_descriptions)
raw.isna().sum()

## 5. Tidy the text

This block makes the station information easier to read and work with.

The first three lines remove unnecessary spaces at the beginning and end of names, descriptions, and source groups. For example, `"Pepper Canyon "` becomes `"Pepper Canyon"`. Spaces between words stay in place.

The code then handles descriptions that contain nothing useful. A description made only of spaces becomes empty after the spaces are removed. `.replace("", pd.NA)` marks that empty value as missing too.

Finally, `.fillna("No details provided")` puts the words **No details provided** wherever a description is missing.

This gives the app something clear to display when the source has no description. Existing descriptions stay as they are, apart from extra spaces at their edges. The block does not create new station details.


In [ ]:
stations["station_name"] = stations["station_name"].str.strip()
stations["description"] = stations["description"].str.strip()
stations["source_group"] = stations["source_group"].str.strip()

# Treat a description containing only spaces as missing too.
stations["description"] = stations["description"].replace("", pd.NA)
stations["description"] = stations["description"].fillna("No details provided")

## 6. Remove repeated copies

This block checks whether the same entry was accidentally listed more than once.

`.duplicated().sum()` counts rows that repeat an earlier row exactly. Here, an exact repeat means all the saved column values match. `.drop_duplicates()` keeps one copy of each repeated row.

Different floors can have the same map location. For example, the AP&M first-floor and second-floor entries share location numbers but have different names. Both stay in the list.

`.reset_index(drop=True)` puts the table's row numbers back in order after any removals. `drop=True` means the old row numbers are discarded instead of becoming another column. These row numbers are only labels for the table.

For this saved list, **no exact repeats were found**, so all 84 entries remain.


In [ ]:
repeated_rows = stations.duplicated().sum()
print("Exact repeated rows:", repeated_rows)

stations = stations.drop_duplicates()
stations = stations.reset_index(drop=True)

print("Entries kept:", len(stations))

## 7. Check names and location numbers

The computer needs readable names and usable location numbers before it can measure distances.

`pd.to_numeric(...)` makes sure the latitude and longitude values are treated as numbers. If a value contains unreadable text, `errors="coerce"` marks it as missing so the checks can catch it.

The block checks that:

1. The table contains at least one entry.
2. Every entry has a name, and no name is blank.
3. Every latitude falls between −90 and 90.
4. Every longitude falls between −180 and 180.

These are the allowed ranges for points on Earth. `.between(...)` checks each number; `.all()` asks whether every value passed.

An `assert` means “this must be true.” If a rule fails, the block stops and shows its message. If all the rules pass, it prints a success message.

Passing these checks means the numbers are usable. It does not prove that a point is on campus or that the station is in the correct building.


In [ ]:
# Change coordinate cells to numbers. Invalid text becomes a missing value.
stations["latitude"] = pd.to_numeric(stations["latitude"], errors="coerce")
stations["longitude"] = pd.to_numeric(stations["longitude"], errors="coerce")

latitude_ok = stations["latitude"].between(-90, 90).all()
longitude_ok = stations["longitude"].between(-180, 180).all()

assert len(stations) > 0, "The station table is empty."
assert stations["station_name"].notna().all(), "A station name is missing."
assert (stations["station_name"] != "").all(), "A station name is blank."
assert latitude_ok and longitude_ok, "Check missing or invalid coordinates."

print("Names and coordinate numbers passed the basic checks.")

## 8. Add source and condition notes

This block tells readers where the information came from and how much is known about current station conditions.

First, it adds the original map link to every entry. It also adds a note saying that the station's current condition and access hours have not been checked.

Next, `.str.contains("closed", ...)` looks for the word **closed** in descriptions. `case=False` makes capital letters and lowercase letters count the same. `na=False` treats a missing description as having no match.

`closure_note` remembers which rows matched. `.loc[closure_note, "status_note"]` selects the notes for those rows and replaces them with a message explaining that the source mentions a closure without a date.

The final line shows those entries for review. The saved result identifies Porters Pub. Its old description is kept, but the note explains that today's condition is unknown. This block reads saved text; it does not check stations live.


In [ ]:
source_url = "https://www.google.com/maps/d/viewer?mid=18OFCg3GFp6wl5mCwioSvU9fdGAA&usp=sharing"
stations["source_url"] = source_url
stations["status_note"] = "Current condition and access hours have not been checked."

# Find an old closure note, without removing the record.
closure_note = stations["description"].str.contains("closed", case=False, na=False)
stations.loc[closure_note, "status_note"] = (
    "Source says closed (no date). Current condition and access are unverified."
)

stations.loc[closure_note, ["station_name", "description", "status_note"]]

## 9. Show the cleanup results

This block puts the counts into one small table so you can see what the cleanup found.

The first calculation works out the share of entries that lacked descriptions:

**31 missing descriptions ÷ 84 entries × 100 = about 36.9%.**

The missing count comes from the original list, before empty descriptions were filled in. That keeps the summary accurate even after the blanks have been labeled.

`pd.DataFrame(...)` creates the summary table. One column names each check; the other gives its result. `round(..., 1)` shows the percentage with one number after the decimal point.

The table shows 84 starting entries, 31 missing descriptions, no exact repeats removed, 84 entries kept, and one old closure note. This step reports the cleanup; it does not change any stations.


In [ ]:
missing_percent = missing_descriptions / original_count * 100

quality = pd.DataFrame({
    "Check": ["Original entries", "Missing descriptions", "Missing descriptions (%)",
              "Exact repeats removed", "Cleaned entries", "Undated closure notes"],
    "Result": [original_count, missing_descriptions, round(missing_percent, 1),
               repeated_rows, len(stations), int(closure_note.sum())]
})
quality

## 10. Set up the nearby-station search

This block prepares a set of instructions called `find_stations`. You give it a starting location, and it gives back the station list ordered from closest to farthest. Defining these instructions lets later blocks reuse them with different starting places.

The steps inside it are:

1. **Check the starting location.** If either number is outside Earth's allowed range, stop with a message asking for a valid location.
2. **Keep the starting point.** `start` holds its latitude and longitude together. `distances = []` makes an empty list for the answers.
3. **Visit each table row.** The `for` line repeats the distance calculation once for each saved station. `station` holds the row being examined at that moment.
4. **Measure the distance.** `geodesic(...)` measures between the starting point and that station's point. `.meters` gives the answer in meters. `.append(...)` adds it to the distance list.
5. **Build the results.** Copy the station table and add a `distance_m` column containing the measured distances.
6. **Order the results.** Put the smallest distances first. When distances match exactly, station names break the tie alphabetically.

`return` gives the finished table back to whichever block requested the search. This block defines the search; the next block actually uses it.

These measurements follow a direct distance across Earth's surface. Walking paths, stairs, entrances, floors, and opening hours are not included. [2, 3]


In [ ]:
def find_stations(latitude, longitude):
    # Reject location numbers outside the allowed range.
    if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
        raise ValueError("Enter a valid latitude and longitude.")

    start = (latitude, longitude)
    distances = []

    # Measure from the starting point to one station at a time.
    for row_number, station in stations.iterrows():
        station_location = (station["latitude"], station["longitude"])
        distance_m = geodesic(start, station_location).meters
        distances.append(distance_m)

    # Add distances to a copy, leaving the station table unchanged.
    results = stations.copy()
    results["distance_m"] = distances
    results = results.sort_values(["distance_m", "station_name"])
    return results.reset_index(drop=True)

## 11. Try the search from one location

This block uses the search with a saved example point near the UC San Diego Central Campus trolley station. The point is approximate. [4]

The four settings mean:

| Setting | Meaning |
| --- | --- |
| `start_latitude` and `start_longitude` | Where this example search begins. |
| `search_radius_m = 500` | Keep stations no more than 500 meters away. |
| `number_to_show = 3` | Show up to three results in the following block. |

The two `assert` lines check that the search distance and number of results are greater than zero.

`find_stations(...)` measures distances and saves the full ordered table as `ranked`. The next line keeps only rows whose distance is **500 meters or less**. `<=` means “less than or equal to.” This smaller table is called `nearby`.

The saved example finds **21 entries within 500 meters**. The next block chooses the three to display. Changing 500 changes the search area; changing 3 changes how many results are shown. The starting point here is typed into the code.


In [ ]:
start_latitude = 32.878353
start_longitude = -117.231842
search_radius_m = 500
number_to_show = 3

assert search_radius_m > 0, "The search distance must be positive."
assert number_to_show > 0, "Show at least one result."

ranked = find_stations(start_latitude, start_longitude)
nearby = ranked[ranked["distance_m"] <= search_radius_m]
print("Mapped entries inside the search distance:", len(nearby))

## 12. Show the closest choices

This block turns the nearby list into a small, readable set of choices.

`nearby.head(number_to_show)` takes the first few rows. With the current setting of 3, it takes up to three. These are the closest entries because the search already placed them in distance order.

`.copy()` keeps this small display table separate. `.round(0)` removes decimal places from its distances, so a distance is shown as a whole number of meters. The earlier full-distance results remain available.

The `if` and `else` lines choose what to show:

- If `top_choices.empty` is true, no choices were found, so show a message suggesting a larger search area or another starting point.
- Otherwise, display each choice's name, distance, description, and condition note.

In the saved example, the closest choices are Matthews Building B, Foodworx, and Dogg House. Their displayed direct distances are about 105, 142, and 144 meters.


In [ ]:
top_choices = nearby.head(number_to_show).copy()
top_choices["distance_m"] = top_choices["distance_m"].round(0)

if top_choices.empty:
    print("No mapped stations in this search area. Increase the distance or choose another point.")
else:
    display(top_choices[["station_name", "distance_m", "description", "status_note"]])

## 13. Draw a distance chart

This block shows the same choices as horizontal bars, making their distances easy to compare. A shorter bar means the station is closer to the example starting point.

`matplotlib.pyplot` supplies the chart tools; `plt` is their short name. If there are any choices to show, the code:

1. Creates a chart area with `figure(...)`.
2. Draws one bar per station with `barh(...)`, using distance to set each bar's length.
3. Puts the closest station at the top with `invert_yaxis()`.
4. Adds the title and a label explaining that distances are in meters.
5. Adjusts spacing with `tight_layout()` so labels fit more comfortably.
6. Saves the picture as `triton_refill_example.png`, then displays it.

The chart uses the rounded example distances from the previous block. If the search found no stations, this block skips drawing a chart.


In [ ]:
import matplotlib.pyplot as plt

if not top_choices.empty:
    plt.figure(figsize=(8, 3.5))
    plt.barh(top_choices["station_name"], top_choices["distance_m"])
    plt.gca().invert_yaxis()
    plt.xlabel("Approximate direct distance (meters)")
    plt.title("Nearby mapped entries from the trolley example point")
    plt.tight_layout()
    plt.savefig("triton_refill_example.png", dpi=160)
    plt.show()

## 14. Check that the search follows its rules

This block runs three checks to help catch mistakes in the search.

First, `check_results = []` starts an empty list where each successful check will be recorded.

1. **Closest first:** Make a list of the distances and compare it with the same numbers arranged from smallest to largest. They should match.
2. **Inside the search area:** Check that every nearby result is within the chosen distance. With the current settings, every result must be 500 meters away or less.
3. **Tiny search area:** Try keeping only stations within one meter of this example starting point. The saved example should produce an empty list.

Each `assert` checks a rule. If it fails, the block stops there. If it passes, `.append(...)` adds the check's name and the word **Passed** to the results list.

The one-meter check is specific to this starting point. Choosing a starting point directly on a station would change the expected answer.


In [ ]:
check_results = []

# The distances should run from smallest to largest.
distance_list = ranked["distance_m"].tolist()
assert distance_list == sorted(distance_list)
check_results.append(["Closest entries come first", "Passed"])

# Every nearby result must be inside the chosen search distance.
assert (nearby["distance_m"] <= search_radius_m).all()
check_results.append(["Nearby results stay inside the search distance", "Passed"])

# This very small search should return no records.
one_meter_results = ranked[ranked["distance_m"] <= 1]
assert one_meter_results.empty
check_results.append(["A one-meter example search returns no results", "Passed"])

## 15. Try situations with predictable answers

This block adds three checks where it is easy to say what should happen.

1. **Start at a saved station.** `stations.iloc[0]` picks the first row; Python starts counting row positions at zero. Its own location becomes the starting point. The closest distance should be essentially zero. The check allows a difference smaller than 0.01 meters.
2. **Start far from campus.** `find_stations(0, 0)` uses a point far away from UCSD. There should be no entries within 500 meters.
3. **Keep separate floors.** `.isin(...)` selects the two named AP&M floor entries. Both should still be present, even though they share a map location. This checks that the earlier cleanup kept them.

Each successful check is added to the list started in the previous block. `pd.DataFrame(...)` turns that list into a table with the check name and result.

The saved table shows **six checks passed**. These checks examine the code and saved list. They do not establish whether a station can be found or used during a campus visit.


In [ ]:
# Start at the exact coordinates of one saved station.
known_station = stations.iloc[0]
at_station = find_stations(known_station["latitude"], known_station["longitude"])
assert at_station.iloc[0]["distance_m"] < 0.01
check_results.append(["A stored station point gives zero distance", "Passed"])

# Search far away from the campus data.
faraway = find_stations(0, 0)
assert faraway[faraway["distance_m"] <= 500].empty
check_results.append(["A faraway test point returns no nearby entries", "Passed"])

# Keep the separate floors even though their coordinates match.
apm_records = stations[stations["station_name"].isin(
    ["AP&M 1st Floor Lobby", "AP&M 2nd Floor Lobby"]
)]
assert len(apm_records) == 2
check_results.append(["Both AP&M floor records are kept", "Passed"])

checks = pd.DataFrame(check_results, columns=["Check", "Result"])
checks

## 16. Save the tables as files

Until now, the tables have been kept inside the running notebook. This block saves them as CSV files, which can be opened by spreadsheet programs and other tools.

`.to_csv(...)` writes a table to the named file. `index=False` leaves out the table's extra row numbers.

| File | Contents |
| --- | --- |
| `triton_refill_stations_rebuild.csv` | The cleaned station list, ready to use in ArcGIS. |
| `triton_refill_data_checks.csv` | Counts from the cleanup. |
| `triton_refill_search_checks.csv` | Results from the six search checks. |
| `triton_refill_example_results.csv` | The example's closest choices and search settings. |

Before saving the example results, the code adds the starting location, the search distance, and a note identifying it as a demonstration. This helps someone reading the file understand how those results were produced.

These files are written in the notebook's current working folder. Running this block again replaces files with the same names there. In Colab, they are saved in the temporary session; the later download block copies the main station file to your computer.


In [ ]:
stations.to_csv("triton_refill_stations_rebuild.csv", index=False)
quality.to_csv("triton_refill_data_checks.csv", index=False)
checks.to_csv("triton_refill_search_checks.csv", index=False)

example_export = top_choices.copy()
example_export["example_start_latitude"] = start_latitude
example_export["example_start_longitude"] = start_longitude
example_export["example_search_radius_m"] = search_radius_m
example_export["example_note"] = "Saved demonstration point; not a live location or walking route."
example_export.to_csv("triton_refill_example_results.csv", index=False)

print("Saved the station list, data checks, search checks and example results.")

## 17. Make a blank sheet for campus visits

This block prepares a place to record real visits to stations later.

`pd.DataFrame(columns=[...])` creates a table with column headings but no filled-in rows. The headings leave room for the visit date, starting place, selected station, whether it was found, whether it worked, travel time, and notes.

For example, after visiting a station you could record “Found it,” “Water was running,” and “Took four minutes.” Those would be observations from the visit. This block does not enter that example or any other results.

The empty table is saved as `triton_refill_field_checks_blank.csv`. It gives future campus testing a consistent place to record what happened. The file produced here contains no completed visits.


In [ ]:
field_checks = pd.DataFrame(columns=[
    "test_number", "visit_date", "starting_place", "station_selected",
    "station_found", "usable_at_visit", "minutes_to_station", "notes"
])
field_checks.to_csv("triton_refill_field_checks_blank.csv", index=False)

print("Created an empty visit log. No field tests have been recorded.")

## 18. Download the cleaned station list

This block copies the main station file from Google Colab to your computer through the browser.

`from google.colab import files` loads Colab's download tool. `files.download(...)` requests the CSV created in step 16: `triton_refill_stations_rebuild.csv`.

Run the earlier blocks first, because the file must exist before it can be downloaded. Once it is on your computer, you can upload it to ArcGIS Online for the station map.

This block is for **Google Colab only**. If you run the notebook in local Jupyter, skip it and open the CSV in the notebook's working folder instead.

This download contains the cleaned station table. Save the notebook itself separately to keep its code and explanations. Uploading the CSV to ArcGIS is also a separate action; running this block does not update the live app. [9, 10]


In [ ]:
# Bring in Colab’s file download tool.
from google.colab import files

# Download the cleaned station CSV to the computer.
files.download("triton_refill_stations_rebuild.csv")

## How the notebook connects to the app

The notebook prepares the station information and demonstrates how a nearby search works. The ArcGIS app uses the published station information to help someone find water on a map.

| Part | Job |
| --- | --- |
| Python notebook | Clean the saved list, check it, try an example search, and save files. |
| ArcGIS Online | Store the published station information and web map. |
| Experience Builder | Provide the app's map, location tools, nearby search, directions, and Street View panel. |
| Survey123 | Collect station reports, notes, and optional photos. |
| Private dashboard | Let the project owner review submitted reports. |

A student can open the app, allow location access or choose a point, view nearby stations, and read the details. They can also get directions or submit a report.

The app has its own search tools; it does not run this notebook when a student opens the map. Editing the saved data in the notebook changes the local example. Updating the live station information requires a separate update in ArcGIS.

[Open Triton Refill](https://experience.arcgis.com/experience/e5cdd600df2146a3a2d6cdd035e79d67).


## What this notebook accomplished

The notebook opened **84 saved entries**, tidied the text, labeled **31 missing descriptions**, and checked for repeated rows and basic location problems. No exact repeats were removed.

It then measured distances from an example trolley location, kept entries within 500 meters, and displayed the three closest choices. Six checks examined whether the search and cleanup behaved as expected.

Finally, it saved the cleaned list, the check results, the example choices, a chart, and an empty sheet for future campus visits.

The distances help compare locations directly. Walking routes, access hours, and whether water is available today require additional information. The saved station list and the code checks alone cannot answer those questions.


## Sources

**[1] Station information.** [UC San Diego Hydration Locations](https://www.google.com/maps/d/viewer?mid=18OFCg3GFp6wl5mCwioSvU9fdGAA&usp=sharing), linked by [UCSD HDH Sustainability](https://hdhsustainability.ucsd.edu/). An earlier map download was turned into the station table saved in this notebook. The source update date is unknown. Current station conditions and access have not been verified.

**[2] Table tools.** pandas documentation for [reading tables](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [ordering rows](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html), [working through rows](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html), and [saving tables](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).

**[3] Measuring distance.** [GeoPy distance documentation](https://geopy.readthedocs.io/en/stable/#calculating-distance). The notebook uses its direct-distance calculation with latitude first and longitude second.

**[4] Example starting place.** [UCSD Stuart Collection: visit via trolley](https://stuartcollection.ucsd.edu/map/visit-via-trolley.html). The numbers 32.878353, −117.231842 were kept from the named Central Campus station point in the page's map during earlier project work. They are an approximate reference; the entrance was not measured or checked in person.

**[5] App map.** [Esri Map widget](https://doc.arcgis.com/en/experience-builder/latest/configure-widgets/map-widget.htm).

**[6] Nearby search in the app.** [Esri Near Me widget](https://doc.arcgis.com/en/experience-builder/latest/configure-widgets/near-me-widget.htm).

**[7] Browser location access.** [W3C Geolocation](https://www.w3.org/TR/geolocation/).

**[8] Putting station data in ArcGIS.** [Esri publishing guide](https://doc.arcgis.com/en/arcgis-online/manage-data/publish-features.htm).

**[9] Using Google Colab.** [Colab FAQ](https://research.google.com/colaboratory/faq.html).

**[10] Downloading a Colab file.** [Colab file tools](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py).

The notebook uses Python, pandas, GeoPy, and Matplotlib. The interactive app was built separately in ArcGIS.
